# Exploratory Data Analysis 

Analysis by Dustin Keate<br>
BSDA Program, WGU

## Data Description

https://www.kaggle.com/datasets/uradkr/s-and-p-500-analyst-rating-and-price-target-accuracy

164,231 sell-side analyst rating actions on S&P 500 stocks from 307 firms, December 2011 to June 2026. Every upgrade, downgrade, and initiation is matched against realized forward stock returns at 30, 90, 180, and 365 days — producing a directional accuracy flag per call. Analysts were right 48.8% of the time at 30 days, rising to 57.0% at one year.

## Packages Used

In [1]:
import pandas as pd
import datetime as dt
from dotenv import load_dotenv
import os
import requests
import json

## Initial Examination

Ingest ```.csv``` data and take an initial look at its size, structure, and completeness.

In [2]:
df = pd.read_csv('../data/sp_500_analyst_rating_and_price_target_accuracy.csv')
df.sample(10)

,ticker,event_date,firm,from_grade,to_grade,action,price_target_action,current_price_target,prior_price_target,implied_direction,forward_return_30d_pct,forward_return_90d_pct,forward_return_180d_pct,forward_return_365d_pct,accuracy_30d,accuracy_90d,accuracy_180d,accuracy_365d
67128,GE,2022-10-26 12:21:51,RBC Capital,NaN,Outperform,main,Raises,93.0,81.0,1,16.803583,37.127696,70.335432,85.317485,1.0,1.0,1.0,1.0
9252,AMAT,2026-05-15 15:44:04,Jefferies,Buy,Buy,main,Raises,510.0,415.0,1,34.329223,NaN,NaN,NaN,1.0,NaN,NaN,NaN
35917,COP,2014-10-21 12:30:00,Jefferies,NaN,Hold,main,Lowers,86.0,89.0,0,0.000000,-8.313970,0.632439,-19.294135,NaN,NaN,NaN,NaN
3395,ACN,2012-06-29 10:09:00,Credit Suisse,NaN,Neutral,main,Lowers,61.0,66.0,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
112638,NXPI,2024-07-12 12:17:00,Oppenheimer,Outperform,Outperform,main,Raises,330.0,295.0,1,-14.286740,-15.338941,-24.478216,-18.339295,0.0,0.0,0.0,0.0
147871,UAL,2016-07-06 10:05:55,Credit Suisse,Outperform,Neutral,down,Lowers,42.0,64.0,-1,23.149321,36.682772,84.965649,97.685058,0.0,0.0,0.0,0.0
83724,JBL,2025-06-18 11:39:49,B of A Securities,Buy,Buy,main,Raises,225.0,180.0,1,9.440041,4.274664,8.169241,81.943443,1.0,1.0,1.0,1.0
49527,DPZ,2013-07-16 10:42:28,JP Morgan,NaN,Neutral,main,Raises,62.0,55.0,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
122143,PNR,2012-10-02 11:37:00,Oppenheimer,NaN,Perform,main,Raises,46.0,45.0,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
16433,AVGO,2024-06-13 15:15:24,UBS,Buy,Buy,main,Raises,1735.0,1610.0,1,2.421284,-5.435687,2.991923,49.935613,1.0,0.0,1.0,1.0


In [3]:
df.shape

(164231, 18)

In [4]:
df.dtypes

ticker                         str
event_date                     str
firm                           str
from_grade                     str
to_grade                       str
action                         str
price_target_action            str
current_price_target       float64
prior_price_target         float64
implied_direction            int64
forward_return_30d_pct     float64
forward_return_90d_pct     float64
forward_return_180d_pct    float64
forward_return_365d_pct    float64
accuracy_30d               float64
accuracy_90d               float64
accuracy_180d              float64
accuracy_365d              float64
dtype: object

In [5]:
df.count()

ticker                     164231
event_date                 164231
firm                       164222
from_grade                  94124
to_grade                   164096
action                     164231
price_target_action        151448
current_price_target       164231
prior_price_target         164231
implied_direction          164231
forward_return_30d_pct     162815
forward_return_90d_pct     158629
forward_return_180d_pct    153242
forward_return_365d_pct    142009
accuracy_30d               116592
accuracy_90d               113769
accuracy_180d              110064
accuracy_365d              102310
dtype: int64

In [6]:
df.nunique()

ticker                        502
event_date                 152609
firm                          306
from_grade                     46
to_grade                       46
action                          5
price_target_action             8
current_price_target         2210
prior_price_target           2185
implied_direction               3
forward_return_30d_pct     100826
forward_return_90d_pct      99076
forward_return_180d_pct     96950
forward_return_365d_pct     92259
accuracy_30d                    2
accuracy_90d                    2
accuracy_180d                   2
accuracy_365d                   2
dtype: int64

Initial Observations:

1. There are ```164231``` entries with 18 different columns consisting of entry metadata, analyst predictions (from and to), ```forward_return``` percentage, and ```accuracy``` assessments.
2. ```event_date``` column is a ```str``` object. The entries will be more valuable as a ```datetime``` object if date becomes relevant.
3. 9 rows are missing the ```firm``` entry.
4. ```forward_return``` percentage drops off naturally as the prediction ```event_date``` is closer to the end date of the data set. Data cannot exist without time travel.
5. It appears an abnormal number of rows are missing ```accuracy``` entries. This warrants additional investigation.


In [7]:
df[df['firm'].isna()]

,ticker,event_date,firm,from_grade,to_grade,action,price_target_action,current_price_target,prior_price_target,implied_direction,forward_return_30d_pct,forward_return_90d_pct,forward_return_180d_pct,forward_return_365d_pct,accuracy_30d,accuracy_90d,accuracy_180d,accuracy_365d
6190,AEP,2014-11-06 10:53:04,NaN,Outperform,Market Perform,down,NaN,0.0,0.0,-1,0.000000,1.749783,-7.802993,-8.394161,0.0,0.0,1.0,1.0
27686,CCL,2016-11-22 23:30:53,NaN,Market Perform,Market Outperform,up,NaN,0.0,0.0,1,1.236219,7.958269,20.612376,32.328741,1.0,1.0,1.0,1.0
45738,DG,2016-10-10 13:06:15,NaN,NaN,Hold,main,Lowers,0.0,0.0,0,3.011064,6.569461,0.919371,18.244725,NaN,NaN,NaN,NaN
70052,GOOG,2012-07-17 13:33:00,NaN,NaN,Outperform,main,Lowers,730.0,750.0,1,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
105166,MU,2015-07-28 17:43:00,NaN,NaN,Buy,up,NaN,0.0,0.0,1,-20.658216,-16.050641,-46.936703,-28.151899,0.0,0.0,0.0,0.0
105561,NCLH,2016-11-22 23:31:58,NaN,Outperform,Market Perform,down,NaN,0.0,0.0,-1,7.776400,19.751555,25.490679,37.416152,0.0,0.0,0.0,0.0
111034,NTRS,2016-10-10 13:06:14,NaN,NaN,Hold,main,Raises,0.0,0.0,0,7.048716,24.113923,19.741670,31.783742,NaN,NaN,NaN,NaN
132928,SNPS,2016-10-10 13:08:22,NaN,NaN,Buy,main,Raises,0.0,0.0,1,-2.542934,-0.792602,17.800520,36.525757,0.0,0.0,1.0,1.0
138842,TEL,2014-01-22 13:00:00,NaN,Hold,Buy,up,Raises,75.0,55.0,1,0.000000,0.000000,0.000000,-0.585740,0.0,0.0,0.0,0.0


Rows that are missing ```firm``` entries are not abnormal except for the identified missing entry. Since the ```firm``` column is a critical element of my hypothesis, remove those rows.

In [8]:
df[df['accuracy_30d'].isna()].sample(10)

,ticker,event_date,firm,from_grade,to_grade,action,price_target_action,current_price_target,prior_price_target,implied_direction,forward_return_30d_pct,forward_return_90d_pct,forward_return_180d_pct,forward_return_365d_pct,accuracy_30d,accuracy_90d,accuracy_180d,accuracy_365d
23232,BR,2026-01-23 15:31:30,DA Davidson,Neutral,Neutral,main,Lowers,228.0,240.0,0,-15.327166,-24.444742,NaN,NaN,NaN,NaN,NaN,NaN
113388,ODFL,2022-04-28 15:06:39,Wells Fargo,NaN,Equal-Weight,main,Raises,280.0,255.0,0,-11.539891,-6.319413,-6.027450,10.113199,NaN,NaN,NaN,NaN
110129,NSC,2016-09-13 11:28:02,Citigroup,NaN,Neutral,main,Raises,97.0,87.0,0,4.496303,22.776675,33.274051,42.843029,NaN,NaN,NaN,NaN
92053,LULU,2025-08-26 14:22:12,Morgan Stanley,Equal-Weight,Equal-Weight,main,Lowers,223.0,280.0,0,-15.069375,-16.224757,-12.057472,NaN,NaN,NaN,NaN,NaN
120481,PHM,2013-02-05 13:33:20,Deutsche Bank,NaN,Hold,main,Raises,17.0,15.0,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
117333,PCAR,2020-03-19 16:56:29,Goldman Sachs,NaN,Neutral,main,Lowers,75.0,85.0,0,19.935709,33.219800,54.342981,73.639604,NaN,NaN,NaN,NaN
66011,FTNT,2021-03-10 15:27:57,BMO Capital,NaN,Market Perform,main,Raises,195.0,180.0,0,9.509031,25.989143,74.044119,58.905028,NaN,NaN,NaN,NaN
117244,PCAR,2023-02-23 11:41:27,UBS,NaN,Neutral,main,Raises,77.0,74.0,0,-3.404122,-3.754447,16.413396,57.856804,NaN,NaN,NaN,NaN
106956,NFLX,2024-01-18 17:54:53,Piper Sandler,Neutral,Neutral,main,Raises,475.0,400.0,0,18.507763,26.453198,35.237284,76.814821,NaN,NaN,NaN,NaN
127931,RL,2017-07-19 11:23:23,Needham,NaN,Hold,init,NaN,0.0,0.0,0,15.554349,18.534832,44.606436,88.994392,NaN,NaN,NaN,NaN


According to this sample, all entries that do not have ```accuracy``` entries are either:
- Missing associated ```forward_return``` entries for reasons already discussed, OR
- Have a ```to_grade``` entry that is associated with a neutral outlook, such as "Equal-Weight," "Hold," or "Neutral."

Both of these are normal and explainable.

**A new observation**: Many rows that have ```event_dates``` prior to approximately 1 Jan 2015 have ```forward_return``` percentages of ```0.0```.

## Initial Data Cleaning

Before performing a more complex analysis, cleaning steps identified above will be completed.

1. Convert ```event_date``` from a ```str``` object to a ```datetime``` object.
2. Remove entries with a missing ```firm``` entry.

In [9]:
df['event_date'] = pd.to_datetime(df['event_date'], format='%Y-%m-%d %H:%M:%S')

df.dropna(subset=['firm'], inplace=True)

## Further Examination of Cleaned Data

In [10]:
df[(df['event_date'] < dt.datetime(2014, 12, 5)) & (df['event_date'] > dt.datetime(2014, 12, 3))][['event_date', 'forward_return_30d_pct', 'accuracy_30d']].sample(10)

,event_date,forward_return_30d_pct,accuracy_30d
16805,2014-12-04 13:36:58,-1.598547,0.0
140646,2014-12-03 05:00:00,0.000000,0.0
112424,2014-12-03 05:00:00,0.000000,0.0
101238,2014-12-04 13:18:46,-1.953922,0.0
142876,2014-12-04 05:00:00,-2.297113,NaN
84165,2014-12-03 14:00:00,0.000000,NaN
16807,2014-12-04 12:55:25,-1.598547,NaN
118951,2014-12-03 05:00:00,0.000000,0.0
20656,2014-12-04 05:00:00,-3.315893,NaN
24354,2014-12-04 05:00:00,-1.465038,0.0


In [11]:
df[df['event_date'] < dt.datetime(2014, 12, 4)].shape

(17723, 18)

Rows with an ```event_date``` on or before ```2014-12-03``` are missing critical ```forward_return``` data that will likely be relevant when performing further analysis.

The original dataset has ```164231``` rows. The missing data affects ```17723``` rows. Removing this will represent a data loss of under 11%.

The dataset contains ratings from December 2011 to June 2026. Of this, December 2011 to November 2014 are missing data. This represents a loss of about 36 months of the 175 months represented, about 20%.

## Further Data Cleaning

Remove rows missing accurate ```forward_return``` entries.

In [12]:
df=df[df['event_date'] > dt.datetime(2014, 12, 4)]
df.shape

(146499, 18)

## S&P 500 Performance Periods

The goal of this section is to identify periods of S&P 500 negative performance as defined by Yardeni Research. This financial consultant describes two types of market losses. One is a market correction which is a loss of 10-20%, and the other is a bear market which is a loss of over 20% (2024).

To define these periods for possible use in my analysis, this requires:

1. Access FMP API @ https://site.financialmodelingprep.com/developer/docs and save data to ```.json``` file to prevent multiple API hits.
2. Load into DataFrame and perform minimal cleaning for analysis.
3. Identify market corrections and bear markets.
4. Create DataFrame and save to ```.json```.

In [13]:
# # This was only run once to get the data from FMP API and save it to a JSON file. The code is commented out to avoid unnecessary API calls.

# load_dotenv()

# FMP_API_KEY = os.getenv("FMP_API_KEY")
# FMP_API_URL = os.getenv("FMP_API_URL")

# url = FMP_API_URL + 'historical-price-eod/light?symbol=^GSPC&from=2000-01-01&to=2026-07-17&apikey=' + FMP_API_KEY
# r = requests.get(url)
# sp_500_data = r.json()

# with open("../data/sp_500_data.json", "w") as file:
#     json.dump(sp_500_data, file, indent=4)

In [14]:
with open("../data/sp_500_data.json") as file:
    sp_500_data = json.load(file)

sp_500_df = pd.DataFrame(sp_500_data)
sp_500_df.drop(columns=['symbol', 'volume'], inplace=True)
sp_500_df['date'] = pd.to_datetime(sp_500_df['date'], format='%Y-%m-%d')

sp_500_df.sample(10)

,date,price
4139,2010-02-01,1089.19
1035,2022-05-31,4132.15
2185,2017-11-02,2579.85
246,2025-07-24,6363.36
3067,2014-05-06,1867.72
2695,2015-10-26,2071.18
3195,2013-10-30,1763.31
3063,2014-05-12,1896.65
4742,2007-09-10,1451.70
4466,2008-10-13,1003.35


In [15]:
local_minima = local_maxima = float(sp_500_df.loc[4999, 'price'])
local_minima_date = local_maxima_date = sp_500_df.loc[4999, 'date'].replace(hour=16, minute=0, second=0)

for i in reversed(range(sp_500_df.shape[0])):
    current_price = float(sp_500_df.loc[i, 'price'])
    current_date = sp_500_df.loc[i, 'date'].replace(hour=16, minute=0, second=0)

    if current_price < local_minima:
        local_minima = current_price
        local_minima_date = current_date
    if current_price > local_maxima:
        percent_loss = (local_maxima - local_minima) / local_maxima
        if percent_loss >= 0.2:
            print(f"Bear Market | Local Maxima: {local_maxima:7.2f}, Local Minima: {local_minima:7.2f}, " +
                  f"Dates: From {local_maxima_date.date()} to {local_minima_date.date()}, Percent Loss: {percent_loss:.1%}")
        elif percent_loss >= 0.1:
            print(f"Correction  | Local Maxima: {local_maxima:7.2f}, Local Minima: {local_minima:7.2f}, " +
                  f"Dates: From {local_maxima_date.date()} to {local_minima_date.date()}, Percent Loss: {percent_loss:.1%}")
        local_maxima = local_minima = current_price
        local_maxima_date = local_minima_date = current_date

Bear Market | Local Maxima: 1565.15, Local Minima:  676.53, Dates: From 2007-10-09 to 2009-03-09, Percent Loss: 56.8%
Correction  | Local Maxima: 2130.82, Local Minima: 1829.08, Dates: From 2015-05-21 to 2016-02-11, Percent Loss: 14.2%
Correction  | Local Maxima: 2872.87, Local Minima: 2581.00, Dates: From 2018-01-26 to 2018-02-08, Percent Loss: 10.2%
Correction  | Local Maxima: 2930.75, Local Minima: 2351.10, Dates: From 2018-09-20 to 2018-12-24, Percent Loss: 19.8%
Bear Market | Local Maxima: 3386.15, Local Minima: 2237.40, Dates: From 2020-02-19 to 2020-03-23, Percent Loss: 33.9%
Bear Market | Local Maxima: 4796.56, Local Minima: 3577.03, Dates: From 2022-01-03 to 2022-10-12, Percent Loss: 25.4%
Correction  | Local Maxima: 6144.14, Local Minima: 4982.78, Dates: From 2025-02-19 to 2025-04-08, Percent Loss: 18.9%


In [ ]:
# This section repeats the previous block, but adds entries to a .json file for future analysis and visualization.

downturns = []

local_minima = local_maxima = float(sp_500_df.loc[4999, 'price'])
local_minima_date = local_maxima_date = sp_500_df.loc[4999, 'date'].replace(hour=16, minute=0, second=0)

for i in reversed(range(sp_500_df.shape[0])):
    current_price = float(sp_500_df.loc[i, 'price'])
    current_date = sp_500_df.loc[i, 'date'].replace(hour=16, minute=0, second=0)

    if current_price < local_minima:
        local_minima = current_price
        local_minima_date = current_date
    if current_price > local_maxima:
        percent_loss = (local_maxima - local_minima) / local_maxima
        if percent_loss >= 0.2:
            downturns.append({
                'Type': 'Bear Market',
                'Local Maxima': local_maxima,
                'Local Minima': local_minima,
                'Start Date': local_maxima_date,
                'End Date': local_minima_date,
                'Percent Loss': percent_loss
            })
        elif percent_loss >= 0.1:
            downturns.append({
                'Type': 'Correction',
                'Local Maxima': local_maxima,
                'Local Minima': local_minima,
                'Start Date': local_maxima_date,
                'End Date': local_minima_date,
                'Percent Loss': percent_loss
            })
        local_maxima = local_minima = current_price
        local_maxima_date = local_minima_date = current_date

downturns_df = pd.DataFrame(downturns)
downturns_df.to_json("../data/sp_500_downturns.json", orient="records", indent=4, date_format="iso")

## Initial Data Exploration

Begin by looking into the stated and approved hypothesis, "Analyst firms classified as optimistic show a higher rate of directional accuracy and forward returns compared to the remaining analyst population." This requires:

1. Define and create metrics for analyst optimism.
2. Identify ways to assess directional accuracy.

## References

Evest. (2025 December 8). Opening price and closing price. Evest Trading Blog. https://www.evest.com/en/trading-blog/opening-price-and-closing-price

Yardeni Research. (2024 January 21). Stock market historical tables: Bull & bear markets. 
https://old.yardeni.com/wp-content/uploads/BullBearTables.pdf